## M1. Actividad 1
Daniel Antonio Melgar Orellana - A00839106
Link GitHub: https://github.com/dmelgar1110/TC2008B.103-Multiagentes

In [1]:
!pip install agentpy
!pip install seaborn

In [2]:
!pip install nbconvert

  Using cached nbconvert-7.16.6-py3-none-any.whl.metadata (8.5 kB)
  Using cached bleach-6.2.0-py3-none-any.whl.metadata (30 kB)
  Using cached defusedxml-0.7.1-py2.py3-none-any.whl.metadata (32 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached jupyterlab_pygments-0.3.0-py3-none-any.whl.metadata (4.4 kB)
  Using cached MarkupSafe-3.0.2-cp313-cp313-win_amd64.whl.metadata (4.1 kB)
  Using cached nbclient-0.10.2-py3-none-any.whl.metadata (8.3 kB)
  Using cached nbformat-5.10.4-py3-none-any.whl.metadata (3.6 kB)
  Using cached pandocfilters-1.5.1-py2.py3-none-any.whl.metadata (9.0 kB)
  Using cached webencodings-0.5.1-py2.py3-none-any.whl.metadata (2.1 kB)
  Using cached tinycss2-1.4.0-py3-none-any.whl.metadata (3.0 kB)
  Using cached fastjsonschema-2.21.2-py3-none-any.whl.metadata (2.3 kB)
  Using cached jsonschema-4.25.1-py3-none-any.whl.metadata (7.6 kB)
  Using cached attrs-25.3.0-py3-none-any.whl.metadata (10 kB)
  Using cached jsonschema_specifications

In [3]:
import numpy as np
import random

Las siguientes entradas definirán los parámetros del agente y el modelo
- Filas y columnas: Para definir el tamaño del tablero.
- Número de agentes: Cantidad de agentes.
- Porcentaje de celdas sucias: Porcentaje de celdas que estarán sucias.
- Tiempo máximo de ejecución: Número máximo de pasos que durará la simulación.


In [4]:


n = int(input("Filas del tablero:"))
m = int(input("Columnas del tablero:"))

nAgentes = int(input("Número agentes:"))
celdasSucias = int(input("Porcentaje de celdas sucias:"))
tiempoMax = int(input('Tiempo máximo de ejecución: '))



- DummyAgent: Es la clase modificada de agente para que puede moverse y limpiar celdas en el entorno. Cada agente lleva un registro de sus movimientos y celdas limpiadas.
- DummyEnvironment: Es la clase modificada que modela el entorno como una cuadrícula donde algunas celdas están sucias y otras limpias. Permite ubicar a los agentes y gestionar el estado de limpieza de cada celda.
- DummyModel: Esta clase casi no fue modificada ya que solo se encarga de controlar la relación de los agentes con el ambiente.

In [ ]:
debug = True
import agentpy as ap
import json

import socket
s=socket.socket(socket.AF_INET, socket.SOCK_STREAM)
s.connect(("127.0.0.1", 1104))
from_server = s.recv(4096)
print ("Received from server: ",from_server.decode("ascii"))
s.send(b"I have received ZZZZ msg$")

'''
A simple agent that moves in a random direction without using a directions array
'''
class DummyAgent(ap.Agent):

    def setup(self):
        # Agent's position. It is regarded as its state.
        self.pos = (0, 0)
        self.movimientos = 0
        self.celdasLimpiadas = 0

    def get_position(self):
        return self.model.environment.positions[self]

    def execute(self):
        #obtener x y del agente
        x, y = self.get_position()
        msg = {
            "type": "step",
            "x": int(x),
            "y": int(y)
        }
        s.send(json.dumps(msg).encode("utf-8"))
        # Si la celda está sucia, limpia
        if not self.model.environment.sucias[x, y]:
            self.model.environment.sucias[x, y] = True
            self.celdasLimpiadas += 1
        else:
            # Generar dx y dy aleatorios entre -1 y 1, pero no puede ser (0,0)
            while True:
                cambioX = random.randint(-1, 1)
                cambioY = random.randint(-1, 1)
                if cambioX != 0 or cambioY != 0:
                    break
            nuevoX = x + cambioX
            nuevoY = y + cambioY
            if 0 <= nuevoX < self.model.environment.shape[0] and 0 <= nuevoY < self.model.environment.shape[1]:
                self.model.environment.move_to(self, (nuevoX, nuevoY))
                self.movimientos += 1


class DummyEnvironment(ap.Grid):
    def setup(self):
        # Matriz de celdas sucias (False = sucia, True = limpia)
        self.sucias = np.full(self.shape, True)
        porcentaje_sucias = self.model.p.porcentaje_sucias
        num_sucias = int(np.prod(self.shape) * porcentaje_sucias / 100)
        contador = 0
        while contador < num_sucias:
            i = random.randint(0, self.shape[0]-1)
            j = random.randint(0, self.shape[1]-1)
            if self.sucias[i, j]:
                self.sucias[i, j] = False
                contador += 1

class DummyModel(ap.Model):

    def setup(self):
        n, m = self.p.tablero
        nAgentes = self.p.nAgentes

        self.environment = DummyEnvironment(self, (n, m))
        self.environment.setup()
        sucias_list = self.environment.sucias.tolist()
        print(sucias_list)
        msg = {
            "type": "setup",
            "sucias": sucias_list
        }
        ##s.send(json.dumps(msg).encode("utf-8"))
        

        self.agentes = ap.AgentList(self, nAgentes, DummyAgent)
        self.environment.add_agents(self.agentes, positions=[(1,1)]*nAgentes)

    def step(self):
        if self.p.print:
            print("********************\n1 Agent's state: \t\t{}\nPosition in environment:{}".format( self.agentes[0].pos, self.environment.positions[self.agentes[0]]))
        
        self.environment.agents.execute()

    def update(self):
        if self.environment.sucias.all():
            self.stop()

# Usar las variables del usuario para los parámetros
#parameters = {
#    'print': False,
#    'tablero': (n, m),
#    'porcentaje_sucias': celdasSucias,
#    'nAgentes': nAgentes,
#    'steps': tiempoMax
#}

parameters = {
    'print': False,
    'tablero': (7, 7),
    'porcentaje_sucias': 50,
    'nAgentes': 1,
    'steps': 500
}

dummyModel = DummyModel(parameters)
result = dummyModel.run()

msg = {"type": "end"}
s.send(json.dumps(msg).encode("utf-8"))
s.send(b"$")  # Opcional, si tu servidor espera el '$' como fin de transmisión
s.close()



total_movimientos = sum(agent.movimientos for agent in dummyModel.agentes)
total_limpiezas = sum(agent.celdasLimpiadas for agent in dummyModel.agentes)
celdas_limpias = np.sum(dummyModel.environment.sucias)
porcentaje_limpias = 100 * celdas_limpias / (parameters['tablero'][0] * parameters['tablero'][1])

print(f"Tiempo transcurrido: {dummyModel.t}")
print(f"Porcentaje de celdas limpias: {porcentaje_limpias:.2f}%")
print(f"Total de movimientos realizados: {total_movimientos}")

[[False, False, False, False, True, True, False], [False, True, False, False, True, True, True], [True, False, False, True, True, False, True], [True, False, True, False, False, False, False], [True, True, True, True, True, False, True], [False, False, True, True, False, True, False], [True, True, False, True, False, True, False]]
Agente en posición (1, 1)
Completed: 1 stepsAgente en posición (0, 2)
Completed: 2 stepsAgente en posición (0, 2)
Completed: 3 stepsAgente en posición (0, 2)
Completed: 4 stepsAgente en posición (0, 2)
Completed: 5 stepsAgente en posición (0, 2)
Completed: 6 stepsAgente en posición (0, 1)
Completed: 7 stepsAgente en posición (0, 1)
Completed: 8 stepsAgente en posición (0, 2)
Completed: 9 stepsAgente en posición (0, 2)
Completed: 10 stepsAgente en posición (0, 2)
Completed: 11 stepsAgente en posición (0, 3)
Completed: 12 stepsAgente en posición (0, 3)
Completed: 13 stepsAgente en posición (0, 3)
Completed: 14 stepsAgente en posición (0, 3)
Completed: 15 stepsA

## Animación del proceso de limpieza
Para la gráfica y animación utilicé el negro para que sea el color de el/los agentes, gris para las celdas sucias y blanco para las limpias.

In [11]:
from IPython.display import HTML
import matplotlib.pyplot as plt
import seaborn as sns

def my_plot(model, ax):
    grid = np.zeros(model.environment.shape)
    # 0 = sucia (gris), 1 = limpia (blanco)
    grid[model.environment.sucias] = 1
    # Mostrar agentes en el grid con valor 2 (negro)
    for agent, pos in model.environment.positions.items():
        grid[pos] = 2
    ax.clear()
    # 0=gris, 1=blanco, 2=negro
    from matplotlib.colors import ListedColormap
    cmap = ListedColormap(['#bdbdbd', '#ffffff', '#000000'])
    ax.imshow(grid, cmap=cmap, vmin=0, vmax=2)
    ax.set_title('Animación limpieza: Gris=sucia, Blanco=limpia, Negro=agente')
    ax.set_xlabel('Columnas')
    ax.set_ylabel('Filas')
    ax.set_xticks([])
    ax.set_yticks([])

dummyModel = DummyModel(parameters)

fig, ax = plt.subplots()
animation = ap.animate(dummyModel, fig, ax, my_plot)
HTML(animation.to_jshtml())

[[True, False, True, True, True, True, False], [True, False, True, False, False, True, True], [True, True, True, False, False, True, True], [True, False, True, True, True, True, False], [True, False, False, True, False, True, False], [False, False, False, False, False, False, False], [False, False, True, True, False, True, False]]
Agente en posición (1, 1)
Agente en posición (1, 1)
Agente en posición (1, 0)
Agente en posición (2, 0)
Agente en posición (2, 0)
Agente en posición (1, 1)
Agente en posición (0, 2)
Agente en posición (0, 3)
Agente en posición (1, 3)
Agente en posición (1, 3)
Agente en posición (2, 2)
Agente en posición (2, 1)
Agente en posición (1, 2)
Agente en posición (2, 2)
Agente en posición (3, 1)
Agente en posición (3, 1)
Agente en posición (2, 1)
Agente en posición (3, 0)
Agente en posición (4, 1)
Agente en posición (4, 1)
Agente en posición (3, 0)
Agente en posición (3, 0)
Agente en posición (3, 1)
Agente en posición (2, 2)
Agente en posición (3, 1)
Agente en posició

## Conclusión y hallazgos

Con esta simulación de multiagente, se logró modelar y visualizar el comportamiento colectivo de agentes simples encargados de limpiar un entorno. Los movimientos de los agentes en este caso no fueron inteligentes, se utilizó una randomización de -1 0 o 1 para elegir cual era su movimiento en x y en y, evitando también que un movimiento sea 0,0 ya que esto significa quedarse parado, y eso no puede pasar.

Luego de haber desarrollado el código y visto la simulaión pude observar que la parametrización del entorno influye directamente en la eficiencia y el tiempo requerido para limpiar el tablero, por el número de agentes, o el porcentaje de celdas sucias, etc.
La animación permite identificar visualmente patrones de movimiento y eficiencia de los agentes, facilitando el análisis del comportamiento colectivo.

Para concluir, esta simulación es un ejemplo que las simulaciones como estas pueden funcionar para analizar sistemas complejos y experimentar con diferentes comportamientos.